# CoraML Facial-Walk Experiment

Train the transformer on facial walks, generate synthetic walks, reconstruct a graph, and compare link-prediction and graph statistics against the true CoraML graph.

## Ready To Run On T4

- This notebook is ready for a `Colab T4 GPU` runtime, not a TPU runtime.
- The training code uses `PyTorch + Hugging Face GPT-2`, so the intended accelerator is `cuda`.
- The graph split is `10% validation + 5% test`, while keeping the training graph connected.
- This cloud config now uses `num_sign_configs=600`, `context_size=16`, `stride=4`, and `n_embd=256` for the next run.
- Checkpoints are written every epoch to `checkpoints/coraml_t4_run`, and the final model is written to `checkpoints/coraml_t4_run/final`.
- If the runtime disconnects, rerunning the training cell with `resume_from_latest=True` resumes from the newest saved epoch.


## Runtime Bootstrap

On Colab/remote kernels, run this first. It can clone the repo onto the remote runtime, switch into it, and mount Drive for persistent checkpoints. Locally it just keeps the current repo root.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = 'https://github.com/rbmuk/facialgen.git'
repo_dir_name = 'facialgen'

running_in_colab = False
drive_root = None

try:
    from google.colab import drive  # type: ignore
    running_in_colab = True
except ImportError:
    drive = None

if running_in_colab:
    runtime_repo_root = Path('/content') / repo_dir_name
    if not runtime_repo_root.exists():
        subprocess.run(['git', 'clone', repo_url, str(runtime_repo_root)], check=True)
    os.chdir(runtime_repo_root)
    print('cwd =', Path.cwd())
    drive.mount('/content/drive')
    drive_root = Path('/content/drive/MyDrive')
    default_save_dir = drive_root / 'facialgen_checkpoints' / 'coraml_t4_run'
else:
    runtime_repo_root = Path.cwd().resolve()
    default_save_dir = runtime_repo_root / 'checkpoints' / 'coraml_t4_run'
    print('cwd =', runtime_repo_root)

default_save_dir = str(default_save_dir)
print('default_save_dir =', default_save_dir)

requirements_path = runtime_repo_root / 'requirements.txt'
if requirements_path.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
    print('Installed dependencies from', requirements_path)
else:
    print('No requirements.txt found at', requirements_path)


cwd = /content/facialgen
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
default_save_dir = /content/drive/MyDrive/facialgen_checkpoints/coraml_t4_run
Installed dependencies from /content/facialgen/requirements.txt


In [20]:
import importlib
from types import SimpleNamespace

import numpy as np
import pandas as pd
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'facialgen').is_dir():
            return candidate
    raise RuntimeError(
        'Could not locate repo root containing pyproject.toml and the facialgen package. '
        'Run the Runtime Bootstrap cell first, and on Colab set repo_url to your GitHub repository.'
    )

repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('repo_root =', repo_root)

import facialgen
import facialgen.data as data_mod
import facialgen.early_stopping as early_stopping
import facialgen.evaluation as evaluation
import facialgen.models as models
import facialgen.train as train_mod

importlib.invalidate_caches()
importlib.reload(data_mod)
importlib.reload(early_stopping)
importlib.reload(evaluation)
importlib.reload(models)
importlib.reload(train_mod)
importlib.reload(facialgen)

from facialgen.data import CyclicFaceChunkDataset
from facialgen.early_stopping import (
    connected_link_prediction_split,
    edge_overlap_ratio,
    link_prediction_scores_from_walks,
    sample_model_walks,
)
from facialgen.evaluation import (
    compute_graph_statistics,
    reconstruct_graph_from_generated_walks,
)
from facialgen.models import FacialGen
from facialgen.train import build_training_objects, resolve_device, seed_everything, train_model

print('CyclicFaceChunkDataset from:', CyclicFaceChunkDataset.__module__)

args = SimpleNamespace(
    dataset_name='coraml',
    seed=2026,
    data_dir='data',
    num_sign_configs=300,
    sign_seed=2026,
    epoch_seed=99,
    context_size=16,
    stride=4,
    batch_size=64,
    epochs=1,
    lr=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    num_workers=0,
    device='auto',
    n_layer=1,
    n_head=4,
    n_embd=256,
    dropout=0.0,
    save_dir=default_save_dir,
    resume_from_latest=True,
    log_every=20,
    early_stop_mode='val',
    early_stop_patience=5,
    early_stop_min_delta=0.0,
    val_fraction=0.10,
    test_fraction=0.05,
    split_seed=123,
    eval_generated_walks=100_000,
    eval_start_mode='random_vertex',
    eval_max_length=None,
    target_edge_overlap=0.5,
    use_link_prediction_split=True,
)

checkpoint_dir = None
num_generated_graphs = 1
final_generated_walks = 500_000
final_start_mode = 'random_vertex'
final_max_length = args.context_size
generation_batch_size = 256
reconstruction_seed = 777

seed_everything(args.seed)
print(f"seed = {args.seed}")

device = resolve_device(args.device)
print(f"dataset = {args.dataset_name}")
print(f"context_size = {args.context_size}, stride = {args.stride}")
print(f"FAST baseline config for CoraML: n_layer = {args.n_layer}")
print(f"checkpoint_dir = {args.save_dir}")
print(f"resume_from_latest = {args.resume_from_latest}")
print('device =', device)


repo_root = /content/facialgen
CyclicFaceChunkDataset from: facialgen.data
seed = 2026
dataset = coraml
context_size = 16, stride = 4
FAST baseline config for CoraML: n_layer = 1
checkpoint_dir = /content/drive/MyDrive/facialgen_checkpoints/coraml_t4_run
resume_from_latest = True
device = cuda


## Reproducibility

Rerunning the import/config cell above now reloads `facialgen` and reseeds Python, NumPy, and PyTorch through `seed_everything(args.seed)`.


In [14]:
print(f"seed = {args.seed}")
print(f"dataset = {args.dataset_name}")
print(f"context_size = {args.context_size}, stride = {args.stride}")
print(f"num_sign_configs = {args.num_sign_configs}")
print(f"eval_generated_walks = {args.eval_generated_walks:,}")
print(f"final_generated_walks = {final_generated_walks:,}")
print(f"save_dir = {args.save_dir}")
print(f"resume_from_latest = {args.resume_from_latest}")
print(f"eval_start_mode = {args.eval_start_mode}")
print(f"final_start_mode = {final_start_mode}")


seed = 2026
dataset = coraml
context_size = 16, stride = 4
num_sign_configs = 300
eval_generated_walks = 100,000
final_generated_walks = 500,000
save_dir = /content/drive/MyDrive/facialgen_checkpoints/coraml_t4_run
resume_from_latest = True


In [9]:
chunk_ds_preview, loader_preview, model_preview, eval_info_preview = build_training_objects(args)
print(f"live stride = {chunk_ds_preview.stride}")
print(f"num full face sequences = {len(chunk_ds_preview.face_dataset)}")
print(f"num chunk samples = {len(chunk_ds_preview)}")


Using connected train split for VAL early stopping: train_edges=6784, val_edges=798, test_edges=399
Dataset: coraml
LCC nodes: 2810
Full face sequences: 43232
Chunk samples @ T=16: 1005633
Chunk stride: 4
Vocab: 2813 (vertices + BOS + EOS + PAD)
live stride = 4
num full face sequences = 43232
num chunk samples = 1005633


In [15]:
if checkpoint_dir is None:
    model, eval_info, history = train_model(args)
else:
    _, _, _, eval_info = build_training_objects(args)
    model = FacialGen.from_pretrained(checkpoint_dir)
    history = []

model.to(device)
print(type(model).__name__)


Using connected train split for VAL early stopping: train_edges=6784, val_edges=798, test_edges=399
Dataset: coraml
LCC nodes: 2810
Full face sequences: 43232
Chunk samples @ T=16: 1005633
Chunk stride: 4
Vocab: 2813 (vertices + BOS + EOS + PAD)


Loading weights:   0%|          | 0/16 [00:00<?, ?it/s]

Resuming from checkpoint: /content/drive/MyDrive/facialgen_checkpoints/coraml_t4_run/epoch_004
Training on device: cuda
Model config: layers=1, heads=4, embd=256, dropout=0.0


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FacialGen


In [16]:
history_df = pd.DataFrame(history)
display(history_df)
if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_df['epoch'], history_df['mean_nll'], marker='o', label='train NLL')
    axes[0].set_title('Training NLL by Epoch')
    axes[0].set_xlabel('epoch')
    axes[0].set_ylabel('mean_nll')
    axes[0].grid(True, alpha=0.3)
    if 'val_roc_auc' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_roc_auc'], marker='o', label='val ROC-AUC')
    if 'val_ap' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_ap'], marker='o', label='val AP')
    if 'val_score' in history_df.columns:
        axes[1].plot(history_df['epoch'], history_df['val_score'], marker='o', label='val score')
    axes[1].set_title('Validation Metrics by Epoch')
    axes[1].set_xlabel('epoch')
    axes[1].set_ylabel('score')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

print('Final model checkpoint:', Path(args.save_dir) / 'final' if args.save_dir else 'not saved')


""


Final model checkpoint: /content/drive/MyDrive/facialgen_checkpoints/coraml_t4_run/final


In [21]:
inspect_checkpoint_dir = checkpoint_dir if checkpoint_dir is not None else (Path(args.save_dir) / 'final' if args.save_dir else None)
inspect_num_samples = 5
inspect_max_length = args.context_size

if inspect_checkpoint_dir is None:
    raise RuntimeError('No checkpoint directory is available to inspect.')

inspect_model = FacialGen.from_pretrained(str(inspect_checkpoint_dir))
inspect_model.to(device)
inspect_walks = sample_model_walks(
    inspect_model,
    num_samples=inspect_num_samples,
    max_length=inspect_max_length,
    bos_token_id=int(eval_info['bos_token_id']),
    eos_token_id=int(eval_info['eos_token_id']),
    device=device,
    start_mode=final_start_mode,
    num_nodes=int(eval_info['num_nodes']),
    batch_size=inspect_num_samples,
    show_progress=False,
)

rows = []
for idx, seq in enumerate(inspect_walks):
    stripped = [tok for tok in seq if tok not in {int(eval_info['bos_token_id']), int(eval_info['eos_token_id'])}]
    rows.append({
        'sample_id': idx,
        'raw_tokens': seq,
        'vertex_tokens': stripped,
        'length_with_specials': len(seq),
        'vertex_length': len(stripped),
    })

display(pd.DataFrame(rows))


Loading weights:   0%|          | 0/16 [00:00<?, ?it/s]

TypeError: sample_model_walks() got an unexpected keyword argument 'start_mode'

In [ ]:
reference_adj = eval_info['reference_adj']
reference_labels = eval_info['reference_labels']
num_nodes = int(eval_info['num_nodes'])
num_reference_edges = int(eval_info['num_reference_edges'])
lp_split = eval_info['link_prediction_split']
if lp_split is None:
    lp_split = connected_link_prediction_split(
        reference_adj,
        val_fraction=args.val_fraction,
        test_fraction=args.test_fraction,
        seed=args.split_seed,
    )

reference_stats = compute_graph_statistics(reference_adj, labels=reference_labels)

generated_results = []
generated_stats_rows = []

for graph_idx in range(num_generated_graphs):
    walks = sample_model_walks(
        model,
        num_samples=final_generated_walks,
        max_length=final_max_length,
        bos_token_id=int(eval_info['bos_token_id']),
        eos_token_id=int(eval_info['eos_token_id']),
        device=device,
        start_mode=final_start_mode,
        num_nodes=num_nodes,
        batch_size=generation_batch_size,
        show_progress=True,
        progress_desc=f'final sampling graph {graph_idx + 1}/{num_generated_graphs}',
    )

    A_hat, S = reconstruct_graph_from_generated_walks(
        walks,
        num_nodes=num_nodes,
        target_num_edges=num_reference_edges,
        seed=reconstruction_seed + graph_idx,
    )

    val_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['val_edges'],
        negative_edges=lp_split['val_non_edges'],
    )
    test_scores = link_prediction_scores_from_walks(
        walks,
        num_nodes=num_nodes,
        positive_edges=lp_split['test_edges'],
        negative_edges=lp_split['test_non_edges'],
    )
    graph_stats = compute_graph_statistics(A_hat, labels=reference_labels)
    overlap = edge_overlap_ratio(A_hat, reference_adj)

    generated_results.append({
        'graph_id': graph_idx,
        'val_roc_auc': float(val_scores['roc_auc']),
        'val_ap': float(val_scores['average_precision']),
        'test_roc_auc': float(test_scores['roc_auc']),
        'test_ap': float(test_scores['average_precision']),
        'edge_overlap': float(overlap),
    })
    generated_stats_rows.append(graph_stats)

lp_table = pd.DataFrame(generated_results)
display(lp_table)

metric_names = list(reference_stats.keys())
stats_table = pd.DataFrame([
    {
        'metric': metric,
        'true_coraml': float(reference_stats[metric]),
        'generated_mean': float(np.nanmean([row[metric] for row in generated_stats_rows])),
        'abs_diff': abs(
            float(np.nanmean([row[metric] for row in generated_stats_rows]))
            - float(reference_stats[metric])
        ),
    }
    for metric in metric_names
])
display(stats_table)

if history:
    display(pd.DataFrame(history))
